# RSNA Knee Super-Consensus Model Training
### Supervision: 0.9159 Macro-AUC Super-Consensus Labels
### Validation Anchor: 58 Gold Adjudicated Studies
### Backbone: CoAtNet RMLP 2 RW 384 with Multi-Instance Learning (MIL) Attention

In [ ]:
!pip install -q timm albumentations
import timm
print("timm version:", timm.__version__)

In [ ]:
# Write training script to train_knee.py
with open("train_knee.py", "w", encoding="utf-8") as f:
    f.write("# %% [code]\n# %% [code]\n#!/usr/bin/env python3\n\"\"\"Knee MRI: training the twelve-finding model\n\nThis is the training half of the model behind the public 0.924 inference notebook. It reads the\nprecomputed slice stacks, trains a CoAtNet backbone with a per-finding attention pooling head, and\nwrites a checkpoint you can drop straight into that notebook.\n\nWHAT YOU NEED ATTACHED\n\n  1. The preprocessed corpus, both parts:\n       kaggle.com/datasets/dreaddevelopment/knee-raptor-corpus          (3,200 studies)\n       kaggle.com/datasets/dreaddevelopment/knee-raptor-corpus-ext      (1,207 studies)\n     Every study is already reduced to a fixed 44 x 336 x 336 uint8 stack, so no DICOM reading\n     happens here. The two parts concatenate in order.\n\n  2. The competition data, for train.csv.\n\n  3. Training labels, as a parquet with a StudyInstanceUID column and the twelve finding columns.\n     THIS IS NOT PROVIDED, and it is the one thing you have to bring yourself. See below.\n\nTHE LABEL PROBLEM, WHICH IS THE REAL PROBLEM\n\nThe competition gives you 4,407 studies and structured labels for only 58 of them. Every other\nstudy carries a free-text radiology report and nothing else. So before any of this trains, you need\nto turn 4,349 reports into twelve numbers each.\n\nThe approach behind the published weights was to read each report with a language model and emit\ntwelve probabilities rather than twelve yes or no answers: a report that hedges, saying a tear is\nsuspected, becomes something near 0.8 rather than a 1. Soft targets are far more forgiving than\nforcing every hedged sentence into a hard label, and the loss here expects them. The 58 studies\nthat come with real labels are held out and used only for validation, never trained on.\n\nPoint --labels at your own parquet built that way. The format is one row per study: a\nStudyInstanceUID column plus the twelve finding columns, values between 0 and 1.\n\nWHAT THE MODEL DOES\n\nThree neighbouring slices are stacked into the three channels of one image, so the network sees a\nlittle of what lies above and below the middle slice: most of the benefit of a 3D model at the cost\nof a 2D one. Each of these three-slice windows goes through the backbone, and the windows are then\npooled by an attention layer that has separate weights for each of the twelve findings. That last\npart matters more than anything else here. A cruciate tear may be visible on two slices while\nosteoarthritis spreads across many, and one shared pooling weight forces those to compete; giving\neach finding its own attention lets each draw on the slices that actually show it.\n\nTraining samples k windows per study at random and evaluates on k_eval windows spread evenly, so\neach epoch sees a different view of the same study. Nothing else is augmented.\n\nAt the end it keeps the best epoch by validation macro-AUC, and also writes a checkpoint that\naverages the weights of the best three epochs. Weight averaging costs nothing at inference, unlike\naveraging predictions from three models, and it usually gives a small gain.\n\nTYPICAL RUN\n\n  python train_knee.py --arch coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k --res 384 --epochs 16\n      --bs 8 --k 12 --k_eval 24 --grad_ckpt --tag mymodel --labels /kaggle/input/YOURS/labels.parquet\n\nAbout three hours on one 4090 for 16 epochs at 384. --grad_ckpt trades a little speed for a lot of\nmemory and is what makes bs 8 fit on a 24 GB card. Use --smoke for a fast wiring check.\n\nMEMORY, MEASURED\n\nThe CoAtNet at 384 with --bs 8 and --grad_ckpt sits at about 23.7 GB, so it needs a 24 GB card.\nOn 16 GB use --bs 2 or 4. Separately from the GPU, this script keeps the best checkpoint plus the\ntop --topk epochs' weights in system RAM so it can average them at the end; that is about 87 MB a\ncopy for the small ViT and 293 MB for the CoAtNet, and the list fills over the first few epochs.\nIf you run out of host memory a few epochs in, set --topk 1 and lower --workers.\n\nThe checkpoint it writes is a dict with keys model, arch, res and lab, which is exactly what the\ninference notebook expects.\n\"\"\"\nimport os, sys, time, json, math, random, argparse\nimport numpy as np, pandas as pd\nimport torch, torch.nn as nn, torch.nn.functional as F\nfrom torch.utils.data import Dataset, DataLoader\nfrom sklearn.metrics import roc_auc_score\nimport timm\n\n# --- ROI localizer (anatomical joint crop). Optional so the no-ROI path is untouched. ---\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\ntry:\n    from roi_localize import square_box as _roi_square_box, compartments as _roi_compartments\nexcept Exception:\n    _roi_square_box = _roi_compartments = None\n\nHERE = os.path.dirname(os.path.abspath(__file__))\nRSNA = os.path.dirname(HERE)\n\n# ---------------------------------------------------------------------------\n# Input discovery. On Kaggle the corpus arrives as two read-only datasets and the\n# competition data as a third, so nothing lives beside this script. Everything in\n# this block is discovery only - the training code further down is unchanged.\n# ---------------------------------------------------------------------------\ndef _find(*names, root=\"/kaggle/input\"):\n    \"\"\"First path under root whose basename matches one of names.\"\"\"\n    for d, _, fs in os.walk(root):\n        for n in names:\n            if n in fs:\n                return os.path.join(d, n)\n    return None\n\n\nclass _TwoPartVols:\n    \"\"\"Presents the two published corpus parts as one array of shape (4407, 44, 336, 336).\n\n    Both parts stay memory-mapped and are never concatenated on disk: copying 22 GB would be\n    pointless when every read is a single study. Row order is part 1 then part 2, matching the\n    order the id files concatenate in. That ordering is the contract between volumes, masks and\n    ids, so do not sort any of them independently.\n    \"\"\"\n    def __init__(self, a, b):\n        self.a, self.b = a, b\n        self.n_a = a.shape[0]\n        self.shape = (a.shape[0] + b.shape[0],) + tuple(a.shape[1:])\n\n    def __len__(self):\n        return self.shape[0]\n\n    def __getitem__(self, row):\n        return self.a[row] if row < self.n_a else self.b[row - self.n_a]\n\n\ndef _open_corpus():\n    \"\"\"Return (vols, masks), from a local single-file corpus or the two public parts.\"\"\"\n    local_v = os.path.join(HERE, \"all_vols.npy\")\n    if os.path.exists(local_v):\n        return (np.load(local_v, mmap_mode=\"r\"),\n                np.load(os.path.join(HERE, \"all_masks.npy\")))\n    av, bv = _find(\"all_vols.npy\"), _find(\"extra_vols.npy\")\n    am, bm = _find(\"all_masks.npy\"), _find(\"extra_masks.npy\")\n    if not all((av, bv, am, bm)):\n        raise SystemExit(\"Could not find the corpus. Attach both parts: \"\n                         \"dreaddevelopment/knee-raptor-corpus and \"\n                         \"dreaddevelopment/knee-raptor-corpus-ext\")\n    vols = _TwoPartVols(np.load(av, mmap_mode=\"r\"), np.load(bv, mmap_mode=\"r\"))\n    masks = np.concatenate([np.load(am), np.load(bm)], axis=0)\n    return vols, masks\n\n\ndef _open_ids():\n    local = os.path.join(HERE, \"all_ids.npy\")\n    if os.path.exists(local):\n        return np.load(local, allow_pickle=True).astype(str)\n    a, b = _find(\"all_ids.npy\"), _find(\"extra_ids.npy\")\n    if not (a and b):\n        raise SystemExit(\"Could not find all_ids.npy / extra_ids.npy - attach both corpus parts.\")\n    return np.concatenate([np.load(a, allow_pickle=True).astype(str),\n                           np.load(b, allow_pickle=True).astype(str)])\nLAB = [\"ACL\",\"MCL\",\"Medial Meniscus\",\"Lateral Meniscus\",\"Medial OA\",\"Lateral OA\",\"PF OA\",\n       \"Effusion\",\"Synovitis\",\"Baker's\",\"Contusion\",\"Fracture\"]\n\n\n_NO_LABELS = \"\"\"\nNo training labels found, so there is nothing to train against.\n\nThe competition labels only 58 of the 4,407 studies. The other 4,349 carry a free-text\nradiology report instead, so before this can train you have to turn those reports into\ntwelve probabilities per study and pass the result with --labels.\n\nExpected format: a parquet with a StudyInstanceUID column plus the columns\n  {cols}\nwith values between 0 and 1. Soft values work better than hard 0/1 here: the loss is built\nfor them, and hedged reports are common.\n\nEverything else in this notebook is ready to run once that file exists.\n\"\"\"\n\n\n# ------------------------------- data ----------------------------------------\nclass StudyWindows(Dataset):\n    \"\"\"Per-study bag of 2.5D windows sampled from all_vols.npy (memmap).\n    Each window = 3 physically-consecutive slices -> RGB, resized to `res`, in [0,1]\n    (matches the SSL input pipeline: ToTensor, no ImageNet norm).\"\"\"\n    def __init__(self, root, ids, id2row, labels, res, k, train, aug=True, norm=\"none\",\n                 roi=False, roi_mode=\"tight\", roi_pad=0.06, roi_overlap=0.12,\n                 roi_sbox=None, roi_cen=None):\n        self.root = root\n        self.ids = ids\n        self.id2row = id2row\n        self.labels = labels              # dict uid -> np.float32[12]\n        self.res, self.k, self.train, self.aug = res, k, train, aug\n        self.norm = norm                  # \"none\"=[0,1] (Raptor SSL); \"imagenet\"=DINOv2 stats\n        self._mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)\n        self._std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)\n        self.vols = None; self.masks = None\n        # --- ROI anatomical joint-crop config ---\n        self.roi = bool(roi)\n        self.roi_mode, self.roi_pad, self.roi_overlap = roi_mode, roi_pad, roi_overlap\n        self.roi_sbox = roi_sbox          # (N,D,4) int16 per-slice tissue bbox, or None\n        self.roi_cen = roi_cen            # (N,D,2) int16 per-slice joint centroid, or None\n        if self.roi and (_roi_square_box is None or roi_sbox is None):\n            raise RuntimeError(\"roi=True but roi_localize or roi_boxes not available\")\n\n    def __len__(self): return len(self.ids)\n\n    def _ensure(self):\n        if self.vols is None:\n            self.vols, self.masks = _open_corpus()   # (N,D,H,W) view, (N,D) u8\n\n    def _centers(self, valid, count):\n        # valid slice indices; window centers must have both neighbors valid & in-range\n        lo, hi = int(valid.min()), int(valid.max())\n        cs = [c for c in range(lo + 1, hi) if c - 1 >= lo and c + 1 <= hi]\n        if not cs: cs = [max(1, min((lo + hi) // 2, self._D - 2))]\n        if self.train:\n            reps = count // len(cs) + 1\n            pool = (cs * reps)\n            random.shuffle(pool)\n            return pool[:count]\n        # eval: evenly spaced deterministic\n        idx = np.linspace(0, len(cs) - 1, count).round().astype(int)\n        return [cs[i] for i in idx]\n\n    def _resize(self, tri):\n        \"\"\"tri: (3,h,w) float32 [0,1] -> (3,res,res) float32.\"\"\"\n        t = torch.from_numpy(np.ascontiguousarray(tri))\n        if t.shape[-1] != self.res or t.shape[-2] != self.res:\n            t = F.interpolate(t[None], size=(self.res, self.res), mode=\"bilinear\",\n                              align_corners=False)[0]\n        return t.numpy()\n\n    def __getitem__(self, i):\n        self._ensure()\n        uid = self.ids[i]; row = self.id2row[uid]\n        self._D = self.vols.shape[1]\n        m = self.masks[row]\n        valid = np.where(m > 0)[0]\n        if len(valid) < 3: valid = np.arange(min(3, self._D))\n        # compartment mode emits 2 crops/center -> sample ceil(k/2) centers to keep #windows==k\n        compart = self.roi and self.roi_mode == \"compartment\"\n        n_centers = (self.k + 1) // 2 if compart else self.k\n        cs = self._centers(valid, n_centers)\n        vol = self.vols[row]  # (D,H,W) u8  (single study read)\n        tiles = []            # list of (3,res,res) float32\n        for c in cs:\n            c = max(1, min(c, self._D - 2))\n            tri = np.stack([vol[c - 1], vol[c], vol[c + 1]], 0).astype(np.float32) / 255.0  # (3,H,W)\n            H, W = tri.shape[-2], tri.shape[-1]\n            if not self.roi:\n                tiles.append(self._resize(tri))\n                continue\n            # one joint box from the CENTER slice, applied to all 3 slices (keeps RGB registered)\n            sq_mode = \"tight\" if compart else self.roi_mode\n            sq = _roi_square_box(tuple(int(v) for v in self.roi_sbox[row, c]), W=W, H=H,\n                                 pad=self.roi_pad, mode=sq_mode,\n                                 centroid=tuple(float(v) for v in self.roi_cen[row, c]))\n            if compart:\n                left, right = _roi_compartments(sq, overlap=self.roi_overlap)\n                for box in (left, right):\n                    x0, y0, x1, y1 = box\n                    tiles.append(self._resize(tri[:, y0:y1, x0:x1]))\n            else:\n                x0, y0, x1, y1 = sq\n                tiles.append(self._resize(tri[:, y0:y1, x0:x1]))\n        if len(tiles) > self.k:\n            tiles = tiles[:self.k]\n        wins = np.stack(tiles, 0)          # (K,3,res,res)\n        x = torch.from_numpy(wins)\n        if self.train and self.aug:\n            # light medical-safe aug: NO flips (laterality is signal); mild intensity jitter\n            g = 1.0 + (random.random() - 0.5) * 0.20\n            x = (x * g).clamp(0, 1)\n        if self.norm == \"imagenet\":        # each backbone at its correct input distribution\n            x = (x - self._mean) / self._std\n        y = torch.from_numpy(self.labels[uid])\n        return x, y\n\n\ndef collate(batch):\n    xs = torch.stack([b[0] for b in batch])   # (B,K,3,res,res)\n    ys = torch.stack([b[1] for b in batch])   # (B,12)\n    return xs, ys\n\n\n# ------------------------------- model ---------------------------------------\ndef build_backbone(arch=\"vit_small_patch16_224\", pretrained=False):\n    hybrid = arch.startswith((\"maxvit\", \"maxxvit\", \"coatnet\", \"coat_\", \"convnext\"))\n    is_vit = (not hybrid) and any(k in arch for k in (\"vit\", \"deit\", \"dinov2\", \"eva\", \"beit\"))\n    kw = dict(pretrained=pretrained, num_classes=0, in_chans=3)\n    if is_vit:\n        kw.update(global_pool=\"token\", dynamic_img_size=True)\n    else:\n        kw.update(global_pool=\"avg\")\n    return timm.create_model(arch, **kw)\n\n\ndef load_raptor(bb, ckpt_path):\n    if ckpt_path in (\"timm\", \"pretrained\"):\n        return \"timm-pretrained\"\n    if ckpt_path in (\"\", \"none\", \"None\"):\n        print(\"[raptor] RANDOM-INIT control (no SSL weights)\", flush=True)\n        return \"random-init\"\n    ck = torch.load(ckpt_path, map_location=\"cpu\", weights_only=False)\n    st = ck[\"student\"] if \"student\" in ck else ck\n    bbst = {k[len(\"backbone.\"):]: v for k, v in st.items() if k.startswith(\"backbone.\")}\n    missing, unexpected = bb.load_state_dict(bbst, strict=False)\n    ep = ck.get(\"epoch\", \"?\")\n    print(f\"[raptor] loaded backbone from {os.path.basename(ckpt_path)} (ssl epoch {ep}) | \"\n          f\"loaded {len(bbst)} tensors, missing {len(missing)}, unexpected {len(unexpected)}\", flush=True)\n    return f\"{os.path.basename(ckpt_path)}@ep{ep}\"\n\n\nclass RaptorClassifier(nn.Module):\n    \"\"\"Raptor encoder + per-diagnosis attention-MIL head (12 findings).\"\"\"\n    def __init__(self, backbone, F_dim=384, n=12, drop=0.2):\n        super().__init__()\n        self.backbone = backbone\n        self.norm = nn.LayerNorm(F_dim)\n        self.att = nn.Sequential(nn.Linear(F_dim, 256), nn.Tanh(), nn.Dropout(drop),\n                                 nn.Linear(256, n))\n        self.clsW = nn.Parameter(torch.zeros(n, F_dim))\n        self.clsb = nn.Parameter(torch.zeros(n))\n        nn.init.trunc_normal_(self.clsW, std=0.02)\n        self.n = n\n\n    def encode(self, x):\n        B, K = x.shape[:2]\n        f = self.backbone(x.flatten(0, 1))\n        return f.view(B, K, -1)\n\n    def head(self, feats):\n        h = self.norm(feats)\n        a = self.att(h)\n        a = torch.softmax(a, dim=1)\n        pooled = torch.einsum(\"bkn,bkf->bnf\", a, h)\n        logits = (pooled * self.clsW).sum(-1) + self.clsb\n        return logits\n\n    def forward(self, x):\n        return self.head(self.encode(x))\n\n\n# ------------------------------- train ---------------------------------------\ndef main():\n    ap = argparse.ArgumentParser()\n    # Defaults to ImageNet-pretrained weights, which is what every released arm was trained from.\n    # The previous default pointed at a self-supervised checkpoint that is not distributed here,\n    # so following the example run above failed on a missing file.\n    ap.add_argument(\"--ckpt\", default=\"timm\",\n                    help=\"'timm' for ImageNet-pretrained weights, or a path to your own \"\n                         \"self-supervised checkpoint\")\n    ap.add_argument(\"--arch\", default=\"vit_small_patch16_224\")\n    ap.add_argument(\"--res\", type=int, default=224)\n    ap.add_argument(\"--k\", type=int, default=12)\n    ap.add_argument(\"--k_eval\", type=int, default=24)\n    ap.add_argument(\"--epochs\", type=int, default=12)\n    ap.add_argument(\"--bs\", type=int, default=8)\n    ap.add_argument(\"--bb_lr\", type=float, default=3e-5)\n    ap.add_argument(\"--head_lr\", type=float, default=1e-3)\n    ap.add_argument(\"--wd\", type=float, default=0.02)\n    ap.add_argument(\"--workers\", type=int, default=4)\n    ap.add_argument(\"--limit\", type=int, default=0)\n    ap.add_argument(\"--freeze_blocks\", type=int, default=0)\n    # Each kept epoch is a full copy of the weights in system RAM (~87 MB small ViT, ~293 MB\n    # CoAtNet). Lower to 1 on a memory-tight machine; at 1 no averaged checkpoint is written.\n    ap.add_argument(\"--topk\", type=int, default=3,\n                    help=\"top epochs kept for weight averaging (1 disables averaging)\")\n    ap.add_argument(\"--norm\", default=\"none\", choices=[\"none\", \"imagenet\"])\n    ap.add_argument(\"--grad_ckpt\", action=\"store_true\")\n    ap.add_argument(\"--tag\", default=\"dev\")\n    ap.add_argument(\"--labels\", default=None,\n                    help=\"parquet of training labels: StudyInstanceUID + the twelve finding \"\n                         \"columns, values 0..1. Not provided with this notebook - see the header.\")\n    ap.add_argument(\"--smoke\", action=\"store_true\")\n    ap.add_argument(\"--seed\", type=int, default=42)\n    # ---- CV fold hook ----\n    ap.add_argument(\"--folds\", type=int, default=0)\n    ap.add_argument(\"--fold\", type=int, default=-1)\n    ap.add_argument(\"--fold_file\", default=None)\n    # ---- anatomical ROI joint-crop (A/B lever). Default OFF -> identical to baseline. ----\n    ap.add_argument(\"--roi\", action=\"store_true\")\n    ap.add_argument(\"--roi_mode\", default=\"compartment\", choices=[\"tight\", \"safe\", \"compartment\"])\n    ap.add_argument(\"--roi_pad\", type=float, default=0.06)\n    ap.add_argument(\"--roi_overlap\", type=float, default=0.12)\n    ap.add_argument(\"--roi_boxes\", default=os.path.join(HERE, \"roi_boxes.npz\"))\n    a = ap.parse_args()\n    random.seed(a.seed); np.random.seed(a.seed); torch.manual_seed(a.seed)\n    dev = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n    if a.smoke:\n        a.epochs, a.bs, a.k, a.k_eval, a.limit, a.workers = 2, 4, 4, 6, 40, 0\n    print(f\"device {dev} | res {a.res} | k {a.k}/{a.k_eval} | bs {a.bs} | tag {a.tag} | \"\n          f\"roi={a.roi}({a.roi_mode})\", flush=True)\n\n    # ---- ids / labels ----\n    ids = _open_ids()\n    id2row = {u: i for i, u in enumerate(ids)}\n    idset = set(ids)\n    _tcsv = os.path.join(RSNA, \"train.csv\")\n    if not os.path.exists(_tcsv):\n        _tcsv = _find(\"train.csv\")\n    if not _tcsv:\n        raise SystemExit(\"Could not find train.csv - attach the competition data.\")\n    tr = pd.read_csv(_tcsv); tr[\"StudyInstanceUID\"] = tr[\"StudyInstanceUID\"].astype(str)\n    gold_df = tr[tr[LAB].notna().all(axis=1)].copy().set_index(\"StudyInstanceUID\")\n    gold_ids = [u for u in gold_df.index if u in idset]\n    _lab = a.labels if (a.labels and os.path.exists(a.labels)) else None\n    if not _lab:\n        _lab = _find(\"SUPER_CONSENSUS_labels.parquet\") or _find(\"SUPER_CONSENSUS_labels.csv\") or _find(\"labels_llm_soft.parquet\")\n    if not _lab:\n        for root, dirs, files in os.walk(\"/kaggle/input\"):\n            for fn in files:\n                if any(k in fn.lower() for k in [\"consensus\", \"soft\", \"label\"]) and fn.endswith((\".parquet\", \".csv\")):\n                    cand = os.path.join(root, fn)\n                    try:\n                        t = pd.read_parquet(cand) if cand.endswith(\".parquet\") else pd.read_csv(cand)\n                        if set(LAB).issubset(t.columns):\n                            _lab = cand\n                            print(f\"[auto-detect] Found matching label file: {_lab}\", flush=True)\n                            break\n                    except Exception:\n                        pass\n            if _lab:\n                break\n    if not _lab or not os.path.exists(_lab):\n        print(f\"DEBUG: Searched /kaggle/input directories: {os.listdir('/kaggle/input') if os.path.exists('/kaggle/input') else 'no /kaggle/input'}\", flush=True)\n        print(_NO_LABELS.format(cols=\", \".join(LAB)), flush=True)\n        raise SystemExit(0)\n    print(f\"[labels] Ingesting training targets from: {_lab}\", flush=True)\n    soft = pd.read_parquet(_lab) if _lab.endswith(\".parquet\") else pd.read_csv(_lab)\n    soft[\"StudyInstanceUID\"] = soft[\"StudyInstanceUID\"].astype(str); soft = soft.set_index(\"StudyInstanceUID\")\n    goldset = set(gold_ids)\n    train_ids = [u for u in ids if u in soft.index and u not in goldset]\n    # ---- CV fold hook: hold out fold `a.fold`, train on the rest ----\n    oof_ids = []\n    if a.folds > 0:\n        assert 0 <= a.fold < a.folds, f\"--fold must be in [0,{a.folds}) when --folds>0\"\n        fmap = json.load(open(a.fold_file))[\"folds\"]\n        held = set(u for u in train_ids if fmap.get(u, -1) == a.fold)\n        oof_ids = [u for u in train_ids if u in held]\n        train_ids = [u for u in train_ids if u not in held]\n        print(f\"[cv] fold {a.fold}/{a.folds}: train {len(train_ids)} | OOF held-out {len(oof_ids)} \"\n              f\"| fold_file {os.path.basename(a.fold_file)}\", flush=True)\n    if a.limit: train_ids = train_ids[:a.limit]\n    labels = {u: soft.loc[u, LAB].values.astype(np.float32) for u in train_ids}\n    for u in oof_ids: labels[u] = soft.loc[u, LAB].values.astype(np.float32)\n    for u in gold_ids: labels[u] = gold_df.loc[u, LAB].values.astype(np.float32)\n    print(f\"train {len(train_ids)} | gold-val {len(gold_ids)}\"\n          + (f\" | oof {len(oof_ids)}\" if oof_ids else \"\"), flush=True)\n\n    prev = np.clip(np.stack([labels[u] for u in train_ids]).mean(0), 0.03, 0.7)\n    pw = torch.tensor(np.clip((1 - prev) / prev, 1, 10), dtype=torch.float32, device=dev)\n\n    # ---- ROI boxes (only when --roi) ----\n    roi_sbox = roi_cen = None\n    if a.roi:\n        rb = np.load(a.roi_boxes)\n        rb_ids = rb[\"ids\"].astype(str)\n        if not np.array_equal(rb_ids, ids):\n            rmap = {u: i for i, u in enumerate(rb_ids)}\n            missing = [u for u in ids if u not in rmap]\n            if missing:\n                raise RuntimeError(f\"roi_boxes missing {len(missing)} corpus ids (e.g. {missing[:2]})\")\n            order = np.array([rmap[u] for u in ids])\n            roi_sbox = rb[\"sbox\"][order]; roi_cen = rb[\"cen\"][order]\n        else:\n            roi_sbox = rb[\"sbox\"]; roi_cen = rb[\"cen\"]\n        print(f\"[roi] ENABLED mode={a.roi_mode} pad={a.roi_pad} overlap={a.roi_overlap} \"\n              f\"boxes={os.path.basename(a.roi_boxes)} sbox={roi_sbox.shape}\", flush=True)\n    _roi_kw = dict(roi=a.roi, roi_mode=a.roi_mode, roi_pad=a.roi_pad, roi_overlap=a.roi_overlap,\n                   roi_sbox=roi_sbox, roi_cen=roi_cen)\n\n    tds = StudyWindows(HERE, train_ids, id2row, labels, a.res, a.k, train=True, norm=a.norm, **_roi_kw)\n    vds = StudyWindows(HERE, gold_ids, id2row, labels, a.res, a.k_eval, train=False, norm=a.norm, **_roi_kw)\n    tl = DataLoader(tds, batch_size=a.bs, shuffle=True, num_workers=a.workers, drop_last=True,\n                    collate_fn=collate, pin_memory=True, persistent_workers=a.workers > 0)\n    vl = DataLoader(vds, batch_size=max(2, a.bs // 2), shuffle=False, num_workers=a.workers,\n                    collate_fn=collate, persistent_workers=a.workers > 0)\n\n    use_timm = a.ckpt in (\"timm\", \"pretrained\")\n    bb = build_backbone(a.arch, pretrained=use_timm)\n    src = load_raptor(bb, a.ckpt)\n    if use_timm: print(f\"[raptor] timm-pretrained backbone: {a.arch}\", flush=True)\n    F_dim = bb.num_features\n    if a.grad_ckpt:\n        try:\n            bb.set_grad_checkpointing(True); print(\"[raptor] gradient checkpointing ON\", flush=True)\n        except Exception as e:\n            print(f\"[raptor] grad_ckpt unsupported for {a.arch}: {e}\", flush=True)\n    model = RaptorClassifier(bb, F_dim=F_dim).to(dev)\n    if a.freeze_blocks > 0:\n        # Backbones name their layer groups differently: ViT-style models use blocks.N., while\n        # CoAtNet, MaxViT and ConvNeXt use stages.N. Matching only one of those froze nothing at\n        # all on the other family, and did it silently. Try both, and always say what happened.\n        pref = tuple([f\"blocks.{b}.\" for b in range(a.freeze_blocks)]\n                     + [f\"stages.{b}.\" for b in range(a.freeze_blocks)])\n        frozen = 0\n        for nm, p in model.backbone.named_parameters():\n            if nm.startswith(pref):\n                p.requires_grad = False\n                frozen += p.numel()\n        n_t = sum(1 for p in model.backbone.parameters() if not p.requires_grad)\n        if frozen:\n            print(f\"[raptor] froze the first {a.freeze_blocks} layer group(s): \"\n                  f\"{n_t} tensors, {frozen/1e6:.2f}M parameters\", flush=True)\n        else:\n            print(f\"[raptor] WARNING: --freeze_blocks {a.freeze_blocks} matched no parameters on \"\n                  f\"{a.arch}; every parameter is still being trained\", flush=True)\n\n    head_params = [p for n_, p in model.named_parameters() if not n_.startswith(\"backbone.\") and p.requires_grad]\n    bb_params = [p for n_, p in model.named_parameters() if n_.startswith(\"backbone.\") and p.requires_grad]\n    opt = torch.optim.AdamW([{\"params\": bb_params, \"lr\": a.bb_lr},\n                             {\"params\": head_params, \"lr\": a.head_lr}], weight_decay=a.wd)\n    steps = max(len(tl) * a.epochs, 1)\n    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[a.bb_lr, a.head_lr],\n                                                total_steps=steps, pct_start=0.15)\n    lossf = nn.BCEWithLogitsLoss(pos_weight=pw)\n\n    @torch.no_grad()\n    def evaluate():\n        model.eval(); P = []; Y = []\n        for x, y in vl:\n            with torch.autocast(dev, dtype=torch.bfloat16, enabled=dev == \"cuda\"):\n                o = torch.sigmoid(model(x.to(dev)).float())\n            P.append(o.cpu().numpy()); Y.append(y.numpy())\n        P = np.concatenate(P); Y = np.concatenate(Y)\n        aucs = {}\n        for j, name in enumerate(LAB):\n            if len(set(Y[:, j].astype(int))) > 1:\n                aucs[name] = float(roc_auc_score(Y[:, j], P[:, j]))\n        return float(np.mean(list(aucs.values()))), aucs, P, Y\n\n    best = 0.0; best_state = None; best_P = None; t0 = time.time(); hist = []\n    TOPK = max(1, a.topk); topk = []\n    for ep in range(a.epochs):\n        model.train(); tot = 0.0\n        for x, y in tl:\n            x, y = x.to(dev, non_blocking=True), y.to(dev, non_blocking=True)\n            opt.zero_grad()\n            with torch.autocast(dev, dtype=torch.bfloat16, enabled=dev == \"cuda\"):\n                logits = model(x)\n                loss = lossf(logits.float(), y)\n            loss.backward()\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 3.0)\n            opt.step(); sched.step(); tot += loss.item()\n        au, aucs, P, Y = evaluate()\n        hist.append({\"ep\": ep, \"loss\": tot / len(tl), \"gold_auc\": au})\n        if au > best:\n            best = au; best_P = P\n            best_state = {\"model\": {k: v.detach().cpu() for k, v in model.state_dict().items()},\n                          \"gold_auc\": au, \"aucs\": aucs, \"src\": src, \"res\": a.res,\n                          \"arch\": a.arch, \"lab\": LAB, \"epoch\": ep}\n        if au >= best and best_state is not None:\n            _tmp = os.path.join(HERE, f\"raptor_ft_{a.tag}.pt.tmp\")\n            torch.save(best_state, _tmp)\n            os.replace(_tmp, os.path.join(HERE, f\"raptor_ft_{a.tag}.pt\"))\n            np.savez(os.path.join(HERE, f\"raptor_gold_{a.tag}.npz\"),\n                     pred=best_P, truth=Y, ids=np.array(gold_ids))\n            json.dump({\"tag\": a.tag, \"src\": src, \"best_gold_auc\": best,\n                       \"aucs\": best_state[\"aucs\"], \"hist\": hist, \"res\": a.res,\n                       \"epochs\": a.epochs, \"bb_lr\": a.bb_lr, \"head_lr\": a.head_lr,\n                       \"n_train\": len(train_ids), \"n_gold\": len(gold_ids),\n                       \"roi\": a.roi, \"roi_mode\": a.roi_mode,\n                       \"partial\": True, \"epochs_done\": ep + 1},\n                      open(os.path.join(HERE, f\"raptor_ft_{a.tag}.json\"), \"w\"), indent=1)\n            print(f\"  [ckpt] best-so-far saved at ep{ep} ({best:.4f})\", flush=True)\n        if len(topk) < TOPK or au > min(t[\"gold_auc\"] for t in topk):\n            topk.append({\"model\": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},\n                         \"gold_auc\": au, \"aucs\": aucs, \"src\": src, \"res\": a.res,\n                         \"arch\": a.arch, \"lab\": LAB, \"epoch\": ep, \"P\": P})\n            topk.sort(key=lambda t: -t[\"gold_auc\"])\n            del topk[TOPK:]\n        print(f\"ep{ep} loss {tot/len(tl):.3f} | GOLD macro-AUC {au:.4f} (best {best:.4f}) | {time.time()-t0:.0f}s\",\n              flush=True)\n    _, aucs, _, Y = evaluate()\n    print(f\"\\nDONE {src} | BEST GOLD macro-AUC {best:.4f}\", flush=True)\n    for k, v in (best_state[\"aucs\"] if best_state else aucs).items():\n        print(f\"   {k:18s} {v:.3f}\", flush=True)\n\n    if best_state is not None:\n        torch.save(best_state, os.path.join(HERE, f\"raptor_ft_{a.tag}.pt\"))\n        np.savez(os.path.join(HERE, f\"raptor_gold_{a.tag}.npz\"),\n                 pred=best_P, truth=Y, ids=np.array(gold_ids))\n        json.dump({\"tag\": a.tag, \"src\": src, \"best_gold_auc\": best, \"aucs\": best_state[\"aucs\"],\n                   \"hist\": hist, \"res\": a.res, \"epochs\": a.epochs, \"bb_lr\": a.bb_lr,\n                   \"head_lr\": a.head_lr, \"n_train\": len(train_ids), \"n_gold\": len(gold_ids),\n                   \"roi\": a.roi, \"roi_mode\": a.roi_mode, \"partial\": False, \"epochs_done\": a.epochs},\n                  open(os.path.join(HERE, f\"raptor_ft_{a.tag}.json\"), \"w\"), indent=1)\n        print(f\"saved raptor_ft_{a.tag}.pt / raptor_gold_{a.tag}.npz / raptor_ft_{a.tag}.json\", flush=True)\n\n    # ---- CV OOF: predict the held-out fold at the best (gold-selected) weights ----\n    if a.folds > 0 and oof_ids and best_state is not None:\n        model.load_state_dict(best_state[\"model\"]); model.eval()\n        ods = StudyWindows(HERE, oof_ids, id2row, labels, a.res, a.k_eval, train=False, norm=a.norm, **_roi_kw)\n        ol = DataLoader(ods, batch_size=max(2, a.bs // 2), shuffle=False, num_workers=a.workers,\n                        collate_fn=collate, persistent_workers=False)\n        Po = []\n        with torch.no_grad():\n            for x, y in ol:\n                with torch.autocast(dev, dtype=torch.bfloat16, enabled=dev == \"cuda\"):\n                    o = torch.sigmoid(model(x.to(dev)).float())\n                Po.append(o.cpu().numpy())\n        Po = np.concatenate(Po)\n        Yo = np.stack([labels[u] for u in oof_ids]).astype(np.float32)\n        np.savez(os.path.join(HERE, f\"raptor_oof_{a.tag}_fold{a.fold}.npz\"),\n                 pred=Po, truth=Yo, ids=np.array(oof_ids), fold=a.fold, nfolds=a.folds)\n        print(f\"[cv] wrote raptor_oof_{a.tag}_fold{a.fold}.npz ({len(oof_ids)} studies; \"\n              f\"truth = soft labels)\", flush=True)\n\n        # --- top-K epoch OOF (new) ---\n        # The epoch-ensemble used to be judged on gold only; that gate is too small to\n        # resolve the move. Re-run the held-out fold at each retained epoch instead.\n        oof_by_ep = {best_state[\"epoch\"]: Po}\n        for t in topk:\n            e = int(t[\"epoch\"])\n            if e in oof_by_ep:\n                continue\n            model.load_state_dict(t[\"model\"]); model.eval()\n            Pe = []\n            with torch.no_grad():\n                for x, y in ol:\n                    with torch.autocast(dev, dtype=torch.bfloat16, enabled=dev == \"cuda\"):\n                        Pe.append(torch.sigmoid(model(x.to(dev)).float()).cpu().numpy())\n            Pe = np.concatenate(Pe); oof_by_ep[e] = Pe\n            np.savez(os.path.join(HERE, f\"raptor_oof_{a.tag}_ep{e}_fold{a.fold}.npz\"),\n                     pred=Pe, truth=Yo, ids=np.array(oof_ids), fold=a.fold, nfolds=a.folds)\n            print(f\"[cv] wrote top-K epoch OOF ep{e}\", flush=True)\n        if len(oof_by_ep) > 1:\n            Pens = np.mean(list(oof_by_ep.values()), axis=0)\n            np.savez(os.path.join(HERE, f\"raptor_oof_{a.tag}_epens_fold{a.fold}.npz\"),\n                     pred=Pens, truth=Yo, ids=np.array(oof_ids), fold=a.fold, nfolds=a.folds)\n            print(f\"[cv] wrote epoch-ensemble OOF over epochs {sorted(oof_by_ep)}\", flush=True)\n        model.load_state_dict(best_state[\"model\"]); model.eval()\n\n    # --- weight-averaged checkpoint (new) ---\n    if len(topk) > 1:\n        import copy\n        sds = [t[\"model\"] for t in topk]\n        avg = {}\n        for k in sds[0]:\n            v0 = sds[0][k]\n            if v0.is_floating_point():\n                avg[k] = sum(sd[k].double() for sd in sds).div(len(sds)).to(v0.dtype)\n            else:\n                avg[k] = v0.clone()          # e.g. num_batches_tracked\n        swa_state = {\"model\": avg, \"gold_auc\": None, \"aucs\": {}, \"src\": src, \"res\": a.res,\n                     \"arch\": a.arch, \"lab\": LAB, \"epoch\": [int(t[\"epoch\"]) for t in topk],\n                     \"swa_over\": [int(t[\"epoch\"]) for t in topk]}\n        model.load_state_dict(avg); model.eval()\n        au_swa, aucs_swa, P_swa, Y_swa = evaluate()\n        swa_state[\"gold_auc\"] = au_swa; swa_state[\"aucs\"] = aucs_swa\n        torch.save(swa_state, os.path.join(HERE, f\"raptor_ft_{a.tag}_swa.pt\"))\n        np.savez(os.path.join(HERE, f\"raptor_gold_{a.tag}_swa.npz\"),\n                 pred=P_swa, truth=Y_swa, ids=np.array(gold_ids))\n        print(f\"SWA over epochs {[int(t['epoch']) for t in topk]} | gold {au_swa:.4f} \"\n              f\"(best-epoch {best:.4f}) {'BETTER' if au_swa > best else 'no gain'}\", flush=True)\n        if a.folds > 0 and oof_ids:\n            ods2 = StudyWindows(HERE, oof_ids, id2row, labels, a.res, a.k_eval, train=False,\n                                norm=a.norm, **_roi_kw)\n            ol2 = DataLoader(ods2, batch_size=max(2, a.bs // 2), shuffle=False,\n                             num_workers=a.workers, collate_fn=collate, persistent_workers=False)\n            Ps = []\n            with torch.no_grad():\n                for x, y in ol2:\n                    with torch.autocast(dev, dtype=torch.bfloat16, enabled=dev == \"cuda\"):\n                        Ps.append(torch.sigmoid(model(x.to(dev)).float()).cpu().numpy())\n            Ps = np.concatenate(Ps)\n            np.savez(os.path.join(HERE, f\"raptor_oof_{a.tag}_swa_fold{a.fold}.npz\"),\n                     pred=Ps, truth=np.stack([labels[u] for u in oof_ids]).astype(np.float32),\n                     ids=np.array(oof_ids), fold=a.fold, nfolds=a.folds)\n            print(f\"[cv] wrote SWA OOF\", flush=True)\n        model.load_state_dict(best_state[\"model\"]); model.eval()\n\n    if len(topk) > 1:\n        eps = [t[\"epoch\"] for t in topk]\n        for rank, t in enumerate(topk):\n            if rank == 0:\n                continue\n            P_t = t.pop(\"P\")\n            np.savez(os.path.join(HERE, f\"raptor_gold_{a.tag}_ep{t['epoch']}.npz\"),\n                     pred=P_t, truth=Y, ids=np.array(gold_ids))\n            t[\"P\"] = P_t\n        Pens = np.mean([t[\"P\"] for t in topk], axis=0)\n        ens_aucs = {}\n        for j, name in enumerate(LAB):\n            if len(set(Y[:, j].astype(int))) > 1:\n                ens_aucs[name] = float(roc_auc_score(Y[:, j], Pens[:, j]))\n        ens = float(np.mean(list(ens_aucs.values())))\n        np.savez(os.path.join(HERE, f\"raptor_gold_{a.tag}_epens.npz\"),\n                 pred=Pens, truth=Y, ids=np.array(gold_ids))\n        print(f\"TOPK epochs {eps} | best {best:.4f} | epoch-ensemble {ens:.4f} \"\n              f\"({'BETTER' if ens > best else 'no gain'})\", flush=True)\n\n\nif __name__ == \"__main__\":\n    main()")
print("train_knee.py ready.")

In [ ]:
import sys
!python train_knee.py \
    --arch coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k \
    --res 384 \
    --epochs 12 \
    --bs 4 \
    --k 12 \
    --k_eval 24 \
    --grad_ckpt \
    --tag super_coatnet_0916 \
    --labels /kaggle/input/rsna-knee-super-consensus-labels-0916/SUPER_CONSENSUS_labels.parquet \
    --workers 2

In [ ]:
!ls -lh raptor_ft_super_coatnet_0916*
import json, glob
for js in glob.glob("raptor_ft_*.json"):
    print("=== " + js + " ===")
    with open(js) as f:
        print(json.dumps(json.load(f), indent=2))